In [1]:
# All the imports

import numpy as np
from skfem import MeshTri, Basis, asm, condense
from skfem.element import ElementTriP1
from skfem.models.poisson import laplace
from skfem.helpers import dot, grad
from skfem import BilinearForm, LinearForm
from skfem.utils import solve
import matplotlib.pyplot as plt
from skfem.visuals.matplotlib import plot
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
import pickle
from scipy.sparse.linalg import splu, LinearOperator, eigsh
import os

In [2]:
# Marking tensors for a 4*4 checkerboard pattern

xs = np.linspace(0, 1, 5)
ys = np.linspace(0, 1, 5)

In [3]:
# Diffusion coefficient for a 4X4 checkerboard pattern.
def D_fun(x, D_max=1.0, D_min=1.0):
    x0 = x[0]; y0 = x[1]
    x_val = ((x0 > 0.25) & (x0 < 0.5)) | ((x0 > 0.75) & (x0 <= 1.0))
    y_val = ((y0 > 0.25) & (y0 < 0.5)) | ((y0 > 0.75) & (y0 <= 1.0))
    same = (x_val == y_val)
    return np.where(same, D_max, D_min)

In [4]:
@BilinearForm
def a(u, v, w):
    return (D_fun(w.x, w.D_max, w.D_min) * dot(grad(u), grad(v))) + (u * v)

In [5]:
@BilinearForm
def mass(u, v, w):
    return u * v

In [6]:
def eigs_at_level(r, D_max=1.0, D_min=1.0):
    m = MeshTri().init_tensor(xs, ys).refined(r)
    basis = Basis(m, ElementTriP1(), intorder=2)
    A = asm(a, basis, D_max=D_max, D_min=D_min)
    M = asm(mass, basis) 

    Boundary_dofs = basis.get_dofs().all()
    A_c, M_c, x0, I = condense(A, M, D=Boundary_dofs)  

    sigma = 0.0
    vals, vecs = eigsh(A_c, k=1, M=M_c, sigma=sigma, which='LM') 

    lambda1 = vals[0]
    u_free = vecs[:, 0]

    u_full = x0.copy()
    u_full[I] = u_free 

    return lambda1, u_full, u_full.size, u_free.size

In [7]:
Range_end = 10

In [9]:
# D_max = 1
if not os.path.exists("Data/Data_dict_dmax_1.pkl"):
    
    Data_dict_dmax_1 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 1
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_1["levels"].append(r)
        Data_dict_dmax_1["lambda1"].append(lambda1)
        Data_dict_dmax_1["full_size"].append(full_size)
        Data_dict_dmax_1["free_size"].append(free_size)
        Data_dict_dmax_1["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_1.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_1, f)

    # Free up memory
    del Data_dict_dmax_1




D_max: 1, level: 0, lambda1: 23.865775936771893, full_size: 25, free_size: 9
D_max: 1, level: 1, lambda1: 21.505544897707868, full_size: 81, free_size: 49
D_max: 1, level: 2, lambda1: 20.92978984221605, full_size: 289, free_size: 225
D_max: 1, level: 3, lambda1: 20.786792290190373, full_size: 1089, free_size: 961
D_max: 1, level: 4, lambda1: 20.75110083703852, full_size: 4225, free_size: 3969
D_max: 1, level: 5, lambda1: 20.74218157149654, full_size: 16641, free_size: 16129
D_max: 1, level: 6, lambda1: 20.739951979525422, full_size: 66049, free_size: 65025
D_max: 1, level: 7, lambda1: 20.73939459541061, full_size: 263169, free_size: 261121
D_max: 1, level: 8, lambda1: 20.739255250213706, full_size: 1050625, free_size: 1046529
D_max: 1, level: 9, lambda1: 20.739220412683174, full_size: 4198401, free_size: 4190209


In [8]:
# D_max = 10

if not os.path.exists("Data/Data_dict_dmax_10.pkl"):
    
    Data_dict_dmax_10 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 10
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_10["levels"].append(r)
        Data_dict_dmax_10["lambda1"].append(lambda1)
        Data_dict_dmax_10["full_size"].append(full_size)
        Data_dict_dmax_10["free_size"].append(free_size)
        Data_dict_dmax_10["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_10.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_10, f)

    # Free up memory
    del Data_dict_dmax_10


D_max: 10, level: 0, lambda1: 126.76176765224538, full_size: 25, free_size: 9
D_max: 10, level: 1, lambda1: 98.47648528942126, full_size: 81, free_size: 49
D_max: 10, level: 2, lambda1: 85.75001011304376, full_size: 289, free_size: 225
D_max: 10, level: 3, lambda1: 79.78830170162708, full_size: 1089, free_size: 961
D_max: 10, level: 4, lambda1: 76.72495780665326, full_size: 4225, free_size: 3969
D_max: 10, level: 5, lambda1: 75.03847902188834, full_size: 16641, free_size: 16129
D_max: 10, level: 6, lambda1: 74.07911657292468, full_size: 66049, free_size: 65025
D_max: 10, level: 7, lambda1: 73.52582203515907, full_size: 263169, free_size: 261121
D_max: 10, level: 8, lambda1: 73.20489995837566, full_size: 1050625, free_size: 1046529
D_max: 10, level: 9, lambda1: 73.01831726364007, full_size: 4198401, free_size: 4190209


In [9]:
# D_max = 20

if not os.path.exists("Data/Data_dict_dmax_20.pkl"):
    
    Data_dict_dmax_20 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 20
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_20["levels"].append(r)
        Data_dict_dmax_20["lambda1"].append(lambda1)
        Data_dict_dmax_20["full_size"].append(full_size)
        Data_dict_dmax_20["free_size"].append(free_size)
        Data_dict_dmax_20["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_20.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_20, f)

    # Free up memory
    del Data_dict_dmax_20

D_max: 20, level: 0, lambda1: 241.09064733610495, full_size: 25, free_size: 9
D_max: 20, level: 1, lambda1: 172.9505157523562, full_size: 81, free_size: 49
D_max: 20, level: 2, lambda1: 142.12524362143304, full_size: 289, free_size: 225
D_max: 20, level: 3, lambda1: 127.90489418796979, full_size: 1089, free_size: 961
D_max: 20, level: 4, lambda1: 119.96707137751801, full_size: 4225, free_size: 3969
D_max: 20, level: 5, lambda1: 114.96068115914309, full_size: 16641, free_size: 16129
D_max: 20, level: 6, lambda1: 111.65215683748144, full_size: 66049, free_size: 65025
D_max: 20, level: 7, lambda1: 109.4309441740057, full_size: 263169, free_size: 261121
D_max: 20, level: 8, lambda1: 107.93144139834168, full_size: 1050625, free_size: 1046529
D_max: 20, level: 9, lambda1: 106.91695055180213, full_size: 4198401, free_size: 4190209


In [10]:
# D_max = 30

if not os.path.exists("Data/Data_dict_dmax_30.pkl"):
    
    Data_dict_dmax_30 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 30
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_30["levels"].append(r)
        Data_dict_dmax_30["lambda1"].append(lambda1)
        Data_dict_dmax_30["full_size"].append(full_size)
        Data_dict_dmax_30["free_size"].append(free_size)
        Data_dict_dmax_30["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_30.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_30, f)

    # Free up memory
    del Data_dict_dmax_30

D_max: 30, level: 0, lambda1: 355.41952701996445, full_size: 25, free_size: 9
D_max: 30, level: 1, lambda1: 236.89947466985564, full_size: 81, free_size: 49
D_max: 30, level: 2, lambda1: 187.42473789210638, full_size: 289, free_size: 225
D_max: 30, level: 3, lambda1: 166.62195562673892, full_size: 1089, free_size: 961
D_max: 30, level: 4, lambda1: 155.0790068742903, full_size: 4225, free_size: 3969
D_max: 30, level: 5, lambda1: 147.39118389775032, full_size: 16641, free_size: 16129
D_max: 30, level: 6, lambda1: 141.93466564689936, full_size: 66049, free_size: 65025
D_max: 30, level: 7, lambda1: 137.99398272497552, full_size: 263169, free_size: 261121
D_max: 30, level: 8, lambda1: 135.13568847967463, full_size: 1050625, free_size: 1046529
D_max: 30, level: 9, lambda1: 133.06028910247886, full_size: 4198401, free_size: 4190209


In [8]:
# D_max = 40

if not os.path.exists("Data/Data_dict_dmax_40.pkl"):
    
    Data_dict_dmax_40 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 40
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_40["levels"].append(r)
        Data_dict_dmax_40["lambda1"].append(lambda1)
        Data_dict_dmax_40["full_size"].append(full_size)
        Data_dict_dmax_40["free_size"].append(free_size)
        Data_dict_dmax_40["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_40.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_40, f)

    # Free up memory
    del Data_dict_dmax_40

D_max: 40, level: 0, lambda1: 469.7484067038241, full_size: 25, free_size: 9
D_max: 40, level: 1, lambda1: 289.37881209980685, full_size: 81, free_size: 49
D_max: 40, level: 2, lambda1: 222.4746121479459, full_size: 289, free_size: 225
D_max: 40, level: 3, lambda1: 197.0600855554259, full_size: 1089, free_size: 961
D_max: 40, level: 4, lambda1: 183.47905554394575, full_size: 4225, free_size: 3969
D_max: 40, level: 5, lambda1: 174.18013305727084, full_size: 16641, free_size: 16129
D_max: 40, level: 6, lambda1: 167.245432454603, full_size: 66049, free_size: 65025
D_max: 40, level: 7, lambda1: 161.97315837130458, full_size: 263169, free_size: 261121
D_max: 40, level: 8, lambda1: 157.95657202539482, full_size: 1050625, free_size: 1046529
D_max: 40, level: 9, lambda1: 154.90039697437885, full_size: 4198401, free_size: 4190209


In [8]:
# D_max = 50

if not os.path.exists("Data/Data_dict_dmax_50.pkl"):
    
    Data_dict_dmax_50 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 50
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_50["levels"].append(r)
        Data_dict_dmax_50["lambda1"].append(lambda1)
        Data_dict_dmax_50["full_size"].append(full_size)
        Data_dict_dmax_50["free_size"].append(free_size)
        Data_dict_dmax_50["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_50.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_50, f)

    # Free up memory
    del Data_dict_dmax_50

D_max: 50, level: 0, lambda1: 584.0772863876832, full_size: 25, free_size: 9
D_max: 50, level: 1, lambda1: 330.6959775217326, full_size: 81, free_size: 49
D_max: 50, level: 2, lambda1: 248.87732394564645, full_size: 289, free_size: 225
D_max: 50, level: 3, lambda1: 220.43220024552275, full_size: 1089, free_size: 961
D_max: 50, level: 4, lambda1: 206.0303250387827, full_size: 4225, free_size: 3969
D_max: 50, level: 5, lambda1: 196.0732016679579, full_size: 16641, free_size: 16129
D_max: 50, level: 6, lambda1: 188.36123692950133, full_size: 66049, free_size: 65025
D_max: 50, level: 7, lambda1: 182.2468027395084, full_size: 263169, free_size: 261121
D_max: 50, level: 8, lambda1: 177.39975632366637, full_size: 1050625, free_size: 1046529
D_max: 50, level: 9, lambda1: 173.57296808230433, full_size: 4198401, free_size: 4190209


In [9]:
# D_max = 60

if not os.path.exists("Data/Data_dict_dmax_60.pkl"):
    
    Data_dict_dmax_60 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 60
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_60["levels"].append(r)
        Data_dict_dmax_60["lambda1"].append(lambda1)
        Data_dict_dmax_60["full_size"].append(full_size)
        Data_dict_dmax_60["free_size"].append(free_size)
        Data_dict_dmax_60["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_60.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_60, f)

    # Free up memory
    del Data_dict_dmax_60

D_max: 60, level: 0, lambda1: 698.406166071543, full_size: 25, free_size: 9
D_max: 60, level: 1, lambda1: 362.3714256239685, full_size: 81, free_size: 49
D_max: 60, level: 2, lambda1: 268.61193333494765, full_size: 289, free_size: 225
D_max: 60, level: 3, lambda1: 238.20101700970486, full_size: 1089, free_size: 961
D_max: 60, level: 4, lambda1: 223.71646059965065, full_size: 4225, free_size: 3969
D_max: 60, level: 5, lambda1: 213.75958849932522, full_size: 16641, free_size: 16129
D_max: 60, level: 6, lambda1: 205.83011591127612, full_size: 66049, free_size: 65025
D_max: 60, level: 7, lambda1: 199.3172075139558, full_size: 263169, free_size: 261121
D_max: 60, level: 8, lambda1: 193.97448205483437, full_size: 1050625, free_size: 1046529
D_max: 60, level: 9, lambda1: 189.62093135281947, full_size: 4198401, free_size: 4190209


In [8]:
# D_max = 70

if not os.path.exists("Data/Data_dict_dmax_70.pkl"):
    
    Data_dict_dmax_70 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 70
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_70["levels"].append(r)
        Data_dict_dmax_70["lambda1"].append(lambda1)
        Data_dict_dmax_70["full_size"].append(full_size)
        Data_dict_dmax_70["free_size"].append(free_size)
        Data_dict_dmax_70["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_70.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_70, f)

    # Free up memory
    del Data_dict_dmax_70

D_max: 70, level: 0, lambda1: 812.7350457554024, full_size: 25, free_size: 9
D_max: 70, level: 1, lambda1: 386.4441536936925, full_size: 81, free_size: 49
D_max: 70, level: 2, lambda1: 283.4724725983511, full_size: 289, free_size: 225
D_max: 70, level: 3, lambda1: 251.75474347204317, full_size: 1089, free_size: 961
D_max: 70, level: 4, lambda1: 237.54965822000958, full_size: 4225, free_size: 3969
D_max: 70, level: 5, lambda1: 227.96277721152958, full_size: 16641, free_size: 16129
D_max: 70, level: 6, lambda1: 220.18816889129369, full_size: 66049, free_size: 65025
D_max: 70, level: 7, lambda1: 213.61410304596296, full_size: 263169, free_size: 261121
D_max: 70, level: 8, lambda1: 208.05882087581895, full_size: 1050625, free_size: 1046529
D_max: 70, level: 9, lambda1: 203.4046721623851, full_size: 4198401, free_size: 4190209


In [73]:
# D_max = 80
if not os.path.exists("Data/Data_dict_dmax_80.pkl"):
    
    Data_dict_dmax_80 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 80
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_80["levels"].append(r)
        Data_dict_dmax_80["lambda1"].append(lambda1)
        Data_dict_dmax_80["full_size"].append(full_size)
        Data_dict_dmax_80["free_size"].append(free_size)
        Data_dict_dmax_80["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_80.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_80, f)

    # Free up memory
    del Data_dict_dmax_80

D_max: 80, level: 0, lambda1: 927.0639254392621, full_size: 25, free_size: 9
D_max: 80, level: 1, lambda1: 404.8365830137606, full_size: 81, free_size: 49
D_max: 80, level: 2, lambda1: 294.8426010171856, full_size: 289, free_size: 225
D_max: 80, level: 3, lambda1: 262.2178093309063, full_size: 1089, free_size: 961
D_max: 80, level: 4, lambda1: 248.4284152650338, full_size: 4225, free_size: 3969
D_max: 80, level: 5, lambda1: 239.37419633858022, full_size: 16641, free_size: 16129
D_max: 80, level: 6, lambda1: 231.96298025801516, full_size: 66049, free_size: 65025
D_max: 80, level: 7, lambda1: 225.55063704487836, full_size: 263169, free_size: 261121
D_max: 80, level: 8, lambda1: 219.99328370048138, full_size: 1050625, free_size: 1046529
D_max: 80, level: 9, lambda1: 215.22274677697, full_size: 4198401, free_size: 4190209


In [72]:
# D_max = 90

if not os.path.exists("Data/Data_dict_dmax_90.pkl"):
    
    Data_dict_dmax_90 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 90
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_90["levels"].append(r)
        Data_dict_dmax_90["lambda1"].append(lambda1)
        Data_dict_dmax_90["full_size"].append(full_size)
        Data_dict_dmax_90["free_size"].append(free_size)
        Data_dict_dmax_90["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_90.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_90, f)

    # Free up memory
    del Data_dict_dmax_90

D_max: 90, level: 0, lambda1: 1041.392805123121, full_size: 25, free_size: 9
D_max: 90, level: 1, lambda1: 419.07965862432627, full_size: 81, free_size: 49
D_max: 90, level: 2, lambda1: 303.71079485037313, full_size: 289, free_size: 225
D_max: 90, level: 3, lambda1: 270.4262700394116, full_size: 1089, free_size: 961
D_max: 90, level: 4, lambda1: 257.07487141770764, full_size: 4225, free_size: 3969
D_max: 90, level: 5, lambda1: 248.59375211833452, full_size: 16641, free_size: 16129
D_max: 90, level: 6, lambda1: 241.63870451616003, full_size: 66049, free_size: 65025
D_max: 90, level: 7, lambda1: 235.51594602258942, full_size: 263169, free_size: 261121
D_max: 90, level: 8, lambda1: 230.09660994246772, full_size: 1050625, free_size: 1046529
D_max: 90, level: 9, lambda1: 225.34557147808047, full_size: 4198401, free_size: 4190209


In [71]:
# D_max = 100

if not os.path.exists("Data/Data_dict_dmax_100.pkl"):
    
    Data_dict_dmax_100 = {"levels": [], "lambda1": [], "full_size": [], "free_size": [], "u_full": [], "elements": [], "chi": []}
    
    D_max = 100
    for r in range(0, Range_end): 
        lambda1, u_full, full_size, free_size = eigs_at_level(r=r, D_max=D_max, D_min=1.0)
        Data_dict_dmax_100["levels"].append(r)
        Data_dict_dmax_100["lambda1"].append(lambda1)
        Data_dict_dmax_100["full_size"].append(full_size)
        Data_dict_dmax_100["free_size"].append(free_size)
        Data_dict_dmax_100["u_full"].append(u_full)
        print(f"D_max: {D_max}, level: {r}, lambda1: {lambda1}, full_size: {full_size}, free_size: {free_size}")

    with open("Data/Data_dict_dmax_100.pkl", "wb") as f:
        pickle.dump(Data_dict_dmax_100, f)

    # Free up memory
    del Data_dict_dmax_100

D_max: 100, level: 0, lambda1: 1155.7216848069806, full_size: 25, free_size: 9
D_max: 100, level: 1, lambda1: 430.29762437837405, full_size: 81, free_size: 49
D_max: 100, level: 2, lambda1: 310.76328760810037, full_size: 289, free_size: 225
D_max: 100, level: 3, lambda1: 276.9781160063967, full_size: 1089, free_size: 961
D_max: 100, level: 4, lambda1: 264.037690343007, full_size: 4225, free_size: 3969
D_max: 100, level: 5, lambda1: 256.1082829126725, full_size: 16641, free_size: 16129
D_max: 100, level: 6, lambda1: 249.63153976473214, full_size: 66049, free_size: 65025
D_max: 100, level: 7, lambda1: 243.8586427979065, full_size: 263169, free_size: 261121
D_max: 100, level: 8, lambda1: 238.6603778104655, full_size: 1050625, free_size: 1046529
D_max: 100, level: 9, lambda1: 234.02056317331107, full_size: 4198401, free_size: 4190209
